In [1]:
#creation clients info

In [2]:
recharge_df=pd.read_csv("/Users/hadilsahraoui/Downloads/backup/projet pfe hadil/recharge_20250311.csv")
achat_df=pd.read_csv("/Users/hadilsahraoui/Downloads/backup/projet pfe hadil/achat_option_20250311.csv")
roue_df=pd.read_csv("/Users/hadilsahraoui/Downloads/backup/projet pfe hadil/roue_chance_20250311.csv")

NameError: name 'pd' is not defined

In [ ]:
recharge_df

In [ ]:
import pandas as pd
recharge_df=pd.read_csv("/Users/hadilsahraoui/Downloads/backup/projet pfe hadil/recharge_20250311.csv")
achat_df=pd.read_csv("/Users/hadilsahraoui/Downloads/backup/projet pfe hadil/achat_option_20250311.csv")
roue_df=pd.read_csv("/Users/hadilsahraoui/Downloads/backup/projet pfe hadil/roue_chance_20250311.csv")
# Ajout des préfixes aux colonnes (comme dans votre code initial)
recharge_df_prefixed = recharge_df.add_prefix('recharge_').rename(columns={'recharge_msisdn': 'msisdn'})
achat_df_prefixed = achat_df.add_prefix('achat_').rename(columns={'achat_msisdn': 'msisdn'})
roue_df_prefixed = roue_df.add_prefix('roue_').rename(columns={'roue_msisdn': 'msisdn'})

df_achat_recharge = pd.merge(
    achat_df, recharge_df,
    on=['msisdn', 'event_date'],
    how='outer',
    suffixes=('_achat', '_recharge')
)

df_final = pd.merge(
    df_achat_recharge, roue_df,
    left_on=['msisdn', 'event_date'],
    right_on=['msisdn', 'entry_date_hist'],
    how='outer'
)


In [ ]:
import pandas as pd
recharge_df=pd.read_csv("/Users/hadilsahraoui/Downloads/backup/projet pfe hadil/recharge_20250311.csv")
achat_df=pd.read_csv("/Users/hadilsahraoui/Downloads/backup/projet pfe hadil/achat_option_20250311.csv")
roue_df=pd.read_csv("/Users/hadilsahraoui/Downloads/backup/projet pfe hadil/roue_chance_20250311.csv")

df_achat_recharge = pd.merge(
    achat_df, recharge_df,
    on=['msisdn', 'event_date'],
    how='outer',
    suffixes=('_achat', '_recharge')
)

final_df= pd.merge(
    df_achat_recharge, roue_df,
    left_on=['msisdn', 'event_date'],
    right_on=['msisdn', 'entry_date_hist'],
    how='outer'
)
# Convertir les colonnes de dates en format datetime si ce n'est pas déjà fait
final_df['recharge_event_date'] = pd.to_datetime(final_df['event_date'])
final_df['achat_event_date'] = pd.to_datetime(final_df['event_date'])
final_df['roue_entry_date_hist'] = pd.to_datetime(final_df['event_date'])

# Extraire le mois et l'année de chaque colonne
final_df['recharge_month'] = final_df['recharge_event_date'].dt.to_period('M')  # Année-Mois
final_df['achat_month'] = final_df['achat_event_date'].dt.to_period('M')  # Année-Mois
final_df['roue_month'] = final_df['roue_entry_date_hist'].dt.to_period('M')  # Année-Mois

import pandas as pd
import re

# Fonction pour extraire la durée en jours de la description
def extract_duree(description):
    description = str(description)
    match = re.search(r'(\d+)\s*(j|jr|jrs)', description, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return 0  # Important de mettre 0 au lieu de None pour convertir ensuite en int

# Appliquer la fonction à chaque ligne de la colonne "achat_description"
final_df['achat_option_duree'] = final_df['description'].apply(extract_duree)

# S'assurer que la colonne est bien en entier
final_df['achat_option_duree'] = final_df['achat_option_duree'].astype(int)

import pandas as pd
import re

# Fonction pour extraire la durée en jours de la description
def extract_duree(description):
    description = str(description)
    match = re.search(r'(\d+)\s*(j|jr|jrs)', description, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return 0  # Important de mettre 0 au lieu de None pour convertir ensuite en int

# Appliquer la fonction à chaque ligne de la colonne "achat_description"
final_df['achat_option_duree'] = final_df['description'].apply(extract_duree)

# S'assurer que la colonne est bien en entier
final_df['achat_option_duree'] = final_df['achat_option_duree'].astype(int)



# Appliquer la fonction à la colonne 'prix'
final_df['prix'] = final_df['prix'].apply(extract_absolute_price)
final_df['amount'] = final_df['amount'].apply(extract_absolute_price)
final_df['recharge_event_date'] = pd.to_datetime(final_df['recharge_event_date'])

# Créer une colonne mois_annee (ex : 01_2025)
final_df['mois_annee'] = final_df['recharge_event_date'].dt.strftime('%m_%Y')

# Créer les DataFrames automatiquement et les nommer dynamiquement
for valeur in final_df['mois_annee'].unique():
    df_temp = final_df[final_df['mois_annee'] == valeur]
    nom_variable = f"df_mois_{valeur}"
    globals()[nom_variable] = df_temp
def detect_type_client(row):
    # Si roue (spin_number) + login_achat → on choisit login_achat
    if pd.notnull(row['spin_number']) and pd.notnull(row['login_achat']):
        return row['login_achat']
    
    # Si roue (spin_number) + login_recharge → on choisit login_recharge
    if pd.notnull(row['spin_number']) and pd.notnull(row['login_recharge']):
        return row['login_recharge']
    
    # Si login_recharge et login_achat sont non-nuls et identiques
    if pd.notnull(row['login_recharge']) and pd.notnull(row['login_achat']):
        if row['login_recharge'] == row['login_achat']:
            return row['login_recharge']

    
    # Si login_recharge seul est non-nul
    if pd.notnull(row['login_recharge']):
        return row['login_recharge']
    
    # Si login_achat seul est non-nul
    if pd.notnull(row['login_achat']):
        return row['login_achat']
    
    # Si seulement roue (spin_number) est présente
    if pd.notnull(row['spin_number']):
        return 'maxit'
    
    # Si aucune condition remplie
    return 'inconnu'


# Appliquer la fonction sur la DataFrame
final_df.loc[:, 'type_client'] = final_df.apply(detect_type_client, axis=1)




final_df['type_client'].unique()

import pandas as pd

# S'assurer que les dates sont au format "jour uniquement"
for col in ['recharge_event_date', 'achat_event_date', 'roue_entry_date_hist']:
    final_df[col] = pd.to_datetime(final_df[col]).dt.date

# Fonction pour appliquer la logique par ligne
def compute_date_entry(row):
    dates = [row['recharge_event_date'], row['achat_event_date'], row['roue_entry_date_hist']]
    dates_present = [d for d in dates if pd.notna(d)]

    if len(dates_present) == 1:
        return dates_present[0]
    elif len(dates_present) == 2:
        return dates_present[0] if dates_present[0] == dates_present[1] else None
    elif len(dates_present) == 3:
        return dates_present[0] if dates_present[0] == dates_present[1] == dates_present[2] else None
    else:
        return None

# Appliquer ligne par ligne
final_df['date_entry'] = final_df.apply(compute_date_entry, axis=1)
final_df['date_entry'] = pd.to_datetime(final_df['date_entry'])
df_global_seg = final_df.groupby("msisdn").agg({
    "type_client": lambda x: x.mode()[0] if not x.mode().empty else 'inconnu',
    'amount': lambda x: (x > 0).sum(),
    'prix': lambda x: (x > 0).sum()
}).reset_index()

def get_nombre_jeu(spin_number):
    if pd.isna(spin_number):
        return 0
    try:
        spin_number = int(spin_number)
        if spin_number >= 0:
            return spin_number
        else:
            return 0
    except (ValueError, TypeError):
        return 0

# Apply the function
final_df['nombre_jeu'] = final_df['spin_number'].apply(get_nombre_jeu)
nombre_jeu_par_client = final_df.groupby('msisdn')['nombre_jeu'].sum()
df_global_seg = df_global_seg.join(nombre_jeu_par_client.rename('nombre_jeu'), on='msisdn')
# 1. Fonction robuste pour détecter le type d'intérêt
def get_interet_type(serv):
    if isinstance(serv, str):
        s = serv.lower()
        if 'data' in s:
            return 'data'
        elif 'voix' in s:
            return 'voix'
        else:
            return 'autre'
    else:
        return 'autre'

final_df['interet'] = final_df['serv_orig'].apply(get_interet_type)

# 2. Fonction robuste pour récupérer le mode
def get_mode(series):
    mode = series.mode()
    if not mode.empty:
        return mode.iloc[0]  # Plus sûr que [0]
    else:
        return 'sans achat'

# 3. Appliquer le mode par msisdn
interet_par_client = final_df.groupby('msisdn')['interet'].apply(get_mode)

# 4. Ajouter au df_global_seg en alignant les index (join)
df_global_seg = df_global_seg.join(interet_par_client.rename('interet'), on='msisdn')
# 1. Créer la colonne d'intérêt international (1 si "roaming" ou "inter" dans serv_orig, sinon 0)
final_df['interet_international'] = final_df['serv_orig'].apply(
    lambda x: 1 if isinstance(x, str) and any(term in x.lower() for term in ['roaming', 'inter']) else 0
)

# 2. Prendre le maximum par client (au moins une occurrence => 1)
interet_international_par_client = final_df.groupby('msisdn')['interet_international'].max()

# 3. Joindre à df_global_seg en gardant l'alignement sur msisdn
df_global_seg = df_global_seg.join(interet_international_par_client.rename('interet_international'), on='msisdn')
df_global_seg['interet_international'] = df_global_seg['interet_international'].fillna(0).astype(int)
df_global_seg.rename(columns={'amount': 'nb_recharge'}, inplace=True)
def get_source_achat(serv):
    """
    Determine the source of purchase based on service string.
    
    Args:
        serv (str): Service string to analyze
        
    Returns:
        str: Source of purchase
    """
    if pd.isna(serv):  # Check for NaN values
        return 'sans achat'
    
    serv = str(serv).lower()  # Convert to string and lowercase for case-insensitive matching
    
    if 'option' in serv:
        return 'option'
    elif 'promo' in serv:
        return 'promo'
    elif 'achat' in serv:
        return 'achat'
    else:
        return 'autre'

# Apply the function with error handling
final_df['source_achat'] = final_df['serv_orig'].apply(get_source_achat)

# 2. Take the maximum per client (at least one occurrence => 1)
source_achat_par_client = final_df.groupby('msisdn')['source_achat'].max()

# Rename the column to avoid conflict before joining
source_achat_par_client = source_achat_par_client.rename('source_achat_max')

# Join with the new column name
df_global_seg = df_global_seg.join(source_achat_par_client, on='msisdn')
import numpy as np
def calculate_final_result(group):
    results = group['sprin_result'].unique()
    
    if any(result not in ['Perdu', np.nan] for result in results):
        return 1
    # If all results are 'Perdu'
    elif all(result == 'Perdu' for result in results):
        return 0
    # If all results are NaN
    elif all(pd.isna(result) for result in results):
        return -1
    # If there's a mix of 'Perdu' and NaN
    else:
        return -1

# Group by MSISDN and apply the calculation
result = final_df.groupby('msisdn').apply(calculate_final_result).reset_index(name='win_count')

# Display the result
df_global_seg = pd.merge(df_global_seg, result, on='msisdn', how='left')

# Fill NaN values with 0 (meaning no win/loss recorded)
df_global_seg['win_count'] = df_global_seg['win_count'].fillna(-1)



df_global_seg.rename(columns={'prix': 'nb_achat'}, inplace=True)
# 1. Calculer la somme des achats par msisdn
montant_total_achat = final_df.groupby('msisdn')['prix'].sum()

# 2. Ajouter la colonne à df_global_seg en alignant sur msisdn
df_global_seg = df_global_seg.join(montant_total_achat.rename('montant_total_achat'), on='msisdn')

# 3. Remplacer les NaN par 0 pour les clients sans achat
df_global_seg['montant_total_achat'] = df_global_seg['montant_total_achat'].fillna(0)
# 1. Calculer la somme des achats par msisdn
montant_total_recharge = final_df.groupby('msisdn')['amount'].sum()

# 2. Ajouter la colonne à df_global_seg en alignant sur msisdn
df_global_seg = df_global_seg.join(montant_total_recharge.rename('montant_total_recharge'), on='msisdn')

# 3. Remplacer les NaN par 0 pour les clients sans achat
df_global_seg['montant_total_recharge'] = df_global_seg['montant_total_recharge'].fillna(0)
df_global_seg['nb_achat'] = df_global_seg['nb_achat'].fillna(0)
df_global_seg['nb_recharge'] = df_global_seg['nb_recharge'].fillna(0)


final_df['interet_promo'] = final_df['serv_orig'].apply(
    lambda x: 1 if isinstance(x, str) and 'promo' in x.lower() else 0
)

interet_promo_par_client = final_df.groupby('msisdn')['interet_promo'].max()
df_global_seg = df_global_seg.join(interet_promo_par_client.rename('interet_promo'), on='msisdn')
df_global_seg['interet_promo'] = df_global_seg['interet_promo'].fillna(0).astype(int)
def get_type_usage(row):
    achat = row['nb_achat'] > 0
    recharge = row['nb_recharge'] > 0
    jeu = row['nombre_jeu'] > 0

    if achat and recharge and jeu:
        return 'complet'
    elif achat and recharge:
        return 'achat_recharge'
    elif achat and jeu:
        return 'achat_jeu'
    elif recharge and jeu:
        return 'recharge_jeu'
    elif achat:
        return 'achat'
    elif recharge:
        return 'recharge'
    elif jeu:
        return 'jeu'
    else:
        return 'aucun'

df_global_seg['type_usage'] = df_global_seg.apply(get_type_usage, axis=1)

def get_action(row):
    achat = row['nb_achat']
    recharge = row['nb_recharge']
    jeu = row['nombre_jeu']
    
    # Trouver la valeur maximale et son type
    max_value = max(achat, recharge, jeu)
    
    if max_value == 0:
        return 'aucun'
    elif max_value == achat:
        return 'achat'
    elif max_value == recharge:
        return 'recharge'
    elif max_value == jeu:
        return 'jeu'
    else:
        return 'complet'

df_global_seg['action_majoritaire'] = df_global_seg.apply(get_action, axis=1)
df_global_seg['montant_total_recharge'] = df_global_seg['montant_total_recharge'].fillna(0)
df_global_seg['montant_total_achat'] = df_global_seg['montant_total_achat'].fillna(0)
# 1. Définir ton score d'engagement
df_global_seg['engagement_score'] = (
    2 * df_global_seg['nb_recharge'] +
    1 * df_global_seg['nb_achat'] +
    3 * df_global_seg['nombre_jeu']
)

# 2. Catégoriser en Low, Medium, High
# Par exemple, baser sur les quantiles (qcut fait des coupures équilibrées)
df_global_seg['engagement_level'] = pd.qcut(
    df_global_seg['engagement_score'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

# 1. Calculer la durée moyenne des options pour chaque client
# Filtrer uniquement les lignes où achat_option_duree est > 0
duree_moyenne_options = final_df[final_df['achat_option_duree'] > 0].groupby('msisdn')['achat_option_duree'].mean()

# 2. Ajouter au df_global_seg
df_global_seg = df_global_seg.join(duree_moyenne_options.rename('duree_moyenne_option'), on='msisdn')

# 3. Remplacer les NaN par 0 pour les clients sans option
df_global_seg['duree_moyenne_option'] = df_global_seg['duree_moyenne_option'].fillna(0)

# 4. Convertir en entier pour éviter les valeurs décimales
df_global_seg['duree_moyenne_option'] = df_global_seg['duree_moyenne_option'].astype(int)
# 1. Fonction pour trouver l'achat majoritaire
def get_major_achat(x):
    mode = x.mode()
    if not mode.empty:
        return mode.iloc[0]
    else:
        return 'aucun achat'

# 2. Créer la colonne achat_serv_orig à partir de serv_orig

# 3. GroupBy sur msisdn
achat_majoritaire = final_df.groupby('msisdn')['description'].apply(get_major_achat)

# 4. Ajouter au df_global_seg en utilisant join pour maintenir l'alignement
df_global_seg = df_global_seg.join(achat_majoritaire.rename('achat_majoritaire'), on='msisdn')

# 5. Remplacer les NaN par 'aucun achat'
df_global_seg['achat_majoritaire'] = df_global_seg['achat_majoritaire'].fillna('aucun achat')
# 1. Calculer dépenses de recharge par client
depenses_recharge = df_global_seg['montant_total_recharge']

# 2. Calculer dépenses d'achat par client
depenses_achat = df_global_seg['montant_total_achat']

# 3. Somme totale des dépenses par client
depenses_totales = depenses_recharge.add(depenses_achat, fill_value=0)

# 4. Trouver le seuil de rentabilité (par exemple médiane)
seuil_rentabilite = depenses_totales.median()

# 5. Créer la colonne rentabilité
df_global_seg['rentabilite'] = depenses_totales.apply(lambda x: 'actif' if x >= seuil_rentabilite else 'passif')
# 1. Nombre total d'achats pour chaque client
total_achats = final_df.groupby('msisdn')['serv_orig'].count()

# 2. Nombre d'achats en promo (en prenant en compte toutes les variations)
achats_promo = final_df[final_df['serv_orig'].str.contains(
    r'promo|_promo|Promo|PROMO', case=False, na=False
)].groupby('msisdn')['serv_orig'].count()

# 3. Calculer le % de réactivité aux promotions et arrondir à 2 décimales
taux_reponse_promo = (achats_promo / total_achats).fillna(0).round(2)

# 4. Ajouter dans ton df global en utilisant join pour maintenir l'alignement
df_global_seg = df_global_seg.join(taux_reponse_promo.rename('reponse_promo'), on='msisdn')

# 5. Remplacer les NaN par 0 pour les clients sans achat
df_global_seg['reponse_promo'] = df_global_seg['reponse_promo'].fillna(0)
# 1. Convertir la date en datetime
final_df['recharge_event_date'] = pd.to_datetime(final_df['recharge_event_date'])

# 2. Trier les données
final_df = final_df.sort_values(by=['msisdn', 'recharge_event_date'])

# 3. Calculer le délai entre les recharges successives
final_df['delai_recharge'] = final_df.groupby('msisdn')['recharge_event_date'].diff()

# 4. Convertir le délai en jours
final_df['delai_recharge'] = final_df['delai_recharge'].dt.total_seconds() / (24 * 60 * 60)

# 5. Calculer le délai moyen entre les recharges pour chaque client
delai_moyen = final_df.groupby('msisdn')['delai_recharge'].mean().round(1)

# 6. Ajouter la colonne du délai moyen dans le dataframe global
df_global_seg = df_global_seg.join(delai_moyen.rename('delai_moyen_recharge'), on='msisdn')

# 7. Remplacer les NaN par -1 pour les clients sans recharge
df_global_seg['delai_moyen_recharge'] = df_global_seg['delai_moyen_recharge'].fillna(-1)

# 1. Convertir les dates en datetime
final_df['event_date'] = pd.to_datetime(final_df['event_date'])





# 4. Extraire uniquement les jours (sans l'heure)
final_df['event_day'] = final_df['event_date'].dt.date

# 5. Compter le nombre de jours distincts d'activité par client
frequence_jours_actifs = final_df.groupby('msisdn')['event_day'].nunique().reset_index()

frequence_jours_actifs = frequence_jours_actifs.rename(columns={'event_day': 'frequence_utilisation_mensuelle'})

# 6. Fusionner avec le DataFrame global
df_global_seg = df_global_seg.merge(frequence_jours_actifs, on='msisdn', how='left')

# 7. Remplacer les NaN par 0 pour les clients inactifs
df_global_seg['frequence_utilisation_mensuelle'] = df_global_seg['frequence_utilisation_mensuelle'].fillna(0).astype(int)
import re

# Fonction pour extraire la quantité consommée en Mo ou Go à partir de la description
def extraire_quantite(description):
    if isinstance(description, str):
        # Recherche des occurrences de Mo ou Go
        match_go = re.search(r'(\d+)\s*Go', description)
        match_mo = re.search(r'(\d+)\s*Mo', description)
        
        if match_go:
            # Si Go est trouvé, convertir en Mo
            return int(match_go.group(1)) * 1000  # 1 Go = 1000 Mo
        elif match_mo:
            # Si Mo est trouvé, retourner directement
            return int(match_mo.group(1))
    return 0  # Retourner 0 si aucune quantité n'est trouvée

# Fonction pour gérer les prix "internet bil milli" (calculer la consommation en Mo)
def calculer_prix(description, prix):
    if isinstance(description, str) and 'internet bil milli' in description.lower():
        # Si "internet bil milli" est dans la description, appliquer le calcul
        return prix * 1000 / 90  # 20 Mo = 90 millimes donc 1 Mo = 90 / 20 millimes
    return 0  # Si aucune correspondance, retour à 0

# Appliquer les fonctions aux données
final_df['quantite_consommée'] = final_df['description'].apply(extraire_quantite)
final_df['prix_internet_bel_milli'] = final_df.apply(lambda row: calculer_prix(row['description'], row['prix']), axis=1)

# Calculer la consommation totale (somme des quantités consommées et des prix)
final_df['consommation_totale_Mo'] = final_df['quantite_consommée'] + final_df['prix_internet_bel_milli']

# Calculer la consommation totale par client
consommation_totale_client = final_df.groupby('msisdn')['consommation_totale_Mo'].sum().reset_index()

# Ajouter cette consommation à df_global_seg
df_global_seg = df_global_seg.merge(consommation_totale_client, on='msisdn', how='left')

# Remplacer les NaN par 0 si nécessaire
df_global_seg['consommation_totale_Mo'] = df_global_seg['consommation_totale_Mo'].fillna(0)

# Afficher les résultats
print(df_global_seg[['msisdn', 'consommation_totale_Mo']].head())

# S'assurer que la colonne date est en datetime
final_df['achat_event_date'] = pd.to_datetime(final_df['event_date'])

# Trier par client et par date d'achat (du plus ancien au plus récent)
final_df = final_df.sort_values(by=['msisdn', 'achat_event_date'])

# Récupérer pour chaque client la dernière offre achetée (description)
derniere_offre = final_df.groupby('msisdn').last()['description'].reset_index()

# Renommer la colonne pour plus de clarté
derniere_offre.rename(columns={'description': 'derniere_offre_achetee'}, inplace=True)

# Fusionner dans df_global_seg
df_global_seg = df_global_seg.merge(derniere_offre, on='msisdn', how='left')


df_global_seg['moy_recharge'] = df_global_seg.apply(lambda row: get_moy_recharge(row['montant_total_recharge'], row['nb_recharge']), axis=1)
   
df_global_seg['est_utilisateur_actif'] = (df_global_seg['frequence_utilisation_mensuelle'] > 0)

df_global_seg


/var/folders/62/qv7mlq7x6r10s2pmbxgzbpzh0000gn/T/ipykernel_31585/2946461303.py:258: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = final_df.groupby('msisdn').apply(calculate_final_result).reset_index(name='win_count')


                                     msisdn  consommation_totale_Mo
0  0000de08121b316e4bcb41aa4b5e4ec8b25159c7                  457290
1  0000e56a98dcfe0de9352da6cc950616aecf9e49                 6202200
2  00019867be5ed9d20a23f82aa0ef736f7488af2f                  777610
3  0002e1afbc4f0398a387d1e2460a58c6d5d278fd                       0
4  0004b3eadb7c0a1be6aea75e06a525532256e159                   25400


,msisdn,type_client,nb_recharge,nb_achat,nombre_jeu,interet,interet_international,source_achat_max,win_count,montant_total_achat,...,duree_moyenne_option,achat_majoritaire,rentabilite,reponse_promo,delai_moyen_recharge,frequence_utilisation_mensuelle,consommation_totale_Mo,derniere_offre_achetee,moy_recharge,est_utilisateur_actif
0,0000de08121b316e4bcb41aa4b5e4ec8b25159c7,srccoption,21,19,0,data,0,sans achat,-1,479300.0,...,27,Navigui 30Go (30j),actif,0.00,2.8,6,457290,Navigui 25Go (30j),30000.0,True
1,0000e56a98dcfe0de9352da6cc950616aecf9e49,srccoption,202,301,22,data,0,sans achat,-1,6182475.0,...,22,Navigui 25Go (30j),actif,0.02,0.3,44,6202200,Navigui 25Go (30j),30000.0,True
2,00019867be5ed9d20a23f82aa0ef736f7488af2f,apirechopt,26,37,18,data,0,sans achat,1,722650.0,...,24,Navigui 25Go (30j),actif,0.03,1.9,10,777610,Navigui 25Go (30j),26250.0,True
3,0002e1afbc4f0398a387d1e2460a58c6d5d278fd,maxit,0,3,0,data,0,option,-1,1625.0,...,0,Internet bil milli,passif,0.00,8.0,3,0,Internet bil milli,0.0,True
4,0004b3eadb7c0a1be6aea75e06a525532256e159,ussd,0,6,0,data,0,promo,-1,27360.0,...,2,Internet bil milli,passif,0.17,6.2,5,25400,Option 25Go avec double validité,0.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75320,fff5cfb5aad8002152f8f874db0a8dbb6497b31c,maxit,12,7,0,autre,0,sans achat,-1,116150.0,...,21,Navigui 25Go (30j),actif,0.00,4.0,4,100530,Navigui 330 Mo (7j),17500.0,True
75321,fff73a17ae58a4b495fc1e6754262de473099db5,apioption,0,5,0,data,0,promo,-1,123500.0,...,0,Option 25Go avec double validité,actif,1.00,17.5,5,125000,Option 25Go avec double validité,0.0,True
75322,fff7ae288cdc5ed12406eb9609d0e6bfbe13d14f,srccoption,6,3,0,autre,0,sans achat,-1,77500.0,...,30,Internet bil milli,actif,0.00,6.5,4,55000,Navigui 30Go (30j),30000.0,True
75323,fff92d87121293906aeffebe9deb70b5ce4da0d2,apioption,0,7,0,data,0,option,-1,47895.0,...,13,Option 400 Mo (2j),passif,0.00,8.2,6,31100,Option 400 Mo (2j),0.0,True


In [ ]:
def detect_maxit_usage(df):
    """
    Détecte si une transaction est MAXIT ou non
    
    Parameters:
    df (pd.DataFrame): Dataframe contenant les données
    
    Returns:
    pd.DataFrame: Dataframe avec une colonne est_maxit
    """
    # Créer une fonction pour vérifier si une ligne est MAXIT
    def is_maxit(row):
        # Vérifier les conditions de MAXIT
        if (pd.notna(row['login_recharge']) and str(row['login_recharge']).lower() == 'maxit') or \
           (pd.notna(row['login_achat']) and str(row['login_achat']).lower() == 'maxit') or \
           (pd.notna(row['spin_number']) and row['spin_number'] > 0):
            return True
        return False
    
    # Appliquer la fonction à chaque ligne
    df['est_maxit'] = df.apply(is_maxit, axis=1)
    
    return df

def extract_duree(description):
    description = str(description)
    match = re.search(r'(\d+)\s*(j|jr|jrs)', description, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return 0  # Important de mettre 0 au lieu de None pour convertir ensuite en int

import matplotlib.pyplot as plt

def detect_type_client(row):
    # Si roue (spin_number) + login_achat → on choisit login_achat
    if pd.notnull(row['spin_number']) and pd.notnull(row['login_achat']):
        return row['login_achat']
    
    # Si roue (spin_number) + login_recharge → on choisit login_recharge
    if pd.notnull(row['spin_number']) and pd.notnull(row['login_recharge']):
        return row['login_recharge']
    
    # Si login_recharge et login_achat sont non-nuls et identiques
    if pd.notnull(row['login_recharge']) and pd.notnull(row['login_achat']):
        if row['login_recharge'] == row['login_achat']:
            return row['login_recharge']

    
    # Si login_recharge seul est non-nul
    if pd.notnull(row['login_recharge']):
        return row['login_recharge']
    
    # Si login_achat seul est non-nul
    if pd.notnull(row['login_achat']):
        return row['login_achat']
    
    # Si seulement roue (spin_number) est présente
    if pd.notnull(row['spin_number']):
        return 'maxit'
    
    # Si aucune condition remplie
    return 'inconnu'


# Appliquer la fonction sur la DataFrame

def detect_maxit_usage(df):
    """
    Détecte si une transaction est MAXIT ou non
    
    Parameters:
    df (pd.DataFrame): Dataframe contenant les données
    
    Returns:
    pd.DataFrame: Dataframe avec une colonne est_maxit
    """
    # Créer une fonction pour vérifier si une ligne est MAXIT
    def is_maxit(row):
        # Vérifier les conditions de MAXIT
        if (pd.notna(row['login_recharge']) and str(row['login_recharge']).lower() == 'maxit') or \
           (pd.notna(row['login_achat']) and str(row['login_achat']).lower() == 'maxit') or \
           (pd.notna(row['spin_number']) and row['spin_number'] > 0):
            return True
        return False
    
    # Appliquer la fonction à chaque ligne
    df['est_maxit'] = df.apply(is_maxit, axis=1)
    
    return df

# Utilisation
def get_nombre_jeu(serv):
    if int(serv) >= 0:
        return int(serv)
    elif pd.isna(serv):
        return 0

def get_interet_type(serv):
    if isinstance(serv, str):
        s = serv.lower()
        if 'data' in s:
            return 'data'
        elif 'voix' in s:
            return 'voix'
        else:
            return 'autre'
    else:
        return 'autre'
def get_mode(series):
    mode = series.mode()
    if not mode.empty:
        return mode.iloc[0]  # Plus sûr que [0]
    else:
        return 'sans achat'
def calculate_final_result(group):
    results = group['sprin_result'].unique()
    
    if any(result not in ['Perdu', np.nan] for result in results):
        return 1
    # If all results are 'Perdu'
    elif all(result == 'Perdu' for result in results):
        return 0
    # If all results are NaN
    elif all(pd.isna(result) for result in results):
        return -1
    # If there's a mix of 'Perdu' and NaN
    else:
        return -1
def check_client_maxit(client_df):
    return pd.Series({'est_maxit': client_df['est_maxit'].any()})
def get_source_achat(serv):
    """
    Determine the source of purchase based on service string.
    
    Args:
        serv (str): Service string to analyze
        
    Returns:
        str: Source of purchase
    """
    if pd.isna(serv):  # Check for NaN values
        return 'sans achat'
    
    serv = str(serv).lower()  # Convert to string and lowercase for case-insensitive matching
    
    if 'option' in serv:
        return 'option'
    elif 'promo' in serv:
        return 'promo'
    elif 'achat' in serv:
        return 'achat'
    else:
        return 'autre'
def get_type_usage(row):
    achat = row['nb_achat'] > 0
    recharge = row['nb_recharge'] > 0
    jeu = row['nombre_jeu'] > 0

    if achat and recharge and jeu:
        return 'complet'
    elif achat and recharge:
        return 'achat_recharge'
    elif achat and jeu:
        return 'achat_jeu'
    elif recharge and jeu:
        return 'recharge_jeu'
    elif achat:
        return 'achat'
    elif recharge:
        return 'recharge'
    elif jeu:
        return 'jeu'
    else:
        return 'aucun'
def get_action(row):
    achat = row['nb_achat']
    recharge = row['nb_recharge']
    jeu = row['nombre_jeu']
    
    # Trouver la valeur maximale et son type
    max_value = max(achat, recharge, jeu)
    
    if max_value == 0:
        return 'aucun'
    elif max_value == achat:
        return 'achat'
    elif max_value == recharge:
        return 'recharge'
    elif max_value == jeu:
        return 'jeu'
    else:
        return 'complet'
def get_major_achat(x):
    mode = x.mode()
    if not mode.empty:
        return mode.iloc[0]
    else:
        return 'aucun achat'
def preparation_profil(client_id,date_debut=None,date_fin=None):
    if date_debut is not None and isinstance(date_debut, str):
        date_debut = pd.to_datetime(date_debut, errors='coerce')
    
    if date_fin is not None and isinstance(date_fin, str):
        date_fin = pd.to_datetime(date_fin, errors='coerce')
    
    recharges_copy = recharge.copy()
    achats_copy = achat.copy()
    spins_copy = spin.copy()
    recharges_copy['event_date'] = pd.to_datetime(recharges_copy['event_date'], format='%Y-%m-%d')
    achats_copy['event_date'] = pd.to_datetime(achats_copy['event_date'], format='%Y-%m-%d')
    spins_copy['entry_date_hist'] = pd.to_datetime(spins_copy['entry_date_hist'], format='%Y-%m-%d')
        
    # Convert date columns to datetime with error handling
    for df in [recharges_copy, achats_copy]:
        df['event_date'] = pd.to_datetime(df['event_date'], errors='coerce')
        
    spins_copy['entry_date_hist'] = pd.to_datetime(spins_copy['entry_date_hist'], errors='coerce')
        
    # Set date range based on inputs
    if date_debut is None or date_fin is None:
            min_dates = [
                recharges_copy['event_date'].min(),
                achats_copy['event_date'].min(),
                spins_copy['entry_date_hist'].min()
            ]
            max_dates = [
                recharges_copy['event_date'].max(),
                achats_copy['event_date'].max(),
                spins_copy['entry_date_hist'].max()
            ]
            date_debut = min(d for d in min_dates if pd.notnull(d))
            date_fin = max(d for d in max_dates if pd.notnull(d))
        
    # Filter the data
    achat_filtered = achats_copy[
        (achats_copy['msisdn'] == client_id) &
        (achats_copy['event_date'].between(date_debut, date_fin, inclusive='both'))
    ].copy()

    rech_filtered = recharges_copy[
        (recharges_copy['msisdn'] == client_id) &
        (recharges_copy['event_date'].between(date_debut, date_fin, inclusive='both'))
    ].copy()

    spin_filtered = spins_copy[
        (spins_copy['msisdn'] == client_id) &
        (spins_copy['entry_date_hist'].between(date_debut, date_fin, inclusive='both'))
    ].copy()
        
    # Ensure we have data to merge
    if achat_filtered.empty and rech_filtered.empty and spin_filtered.empty:
        return pd.DataFrame()  # Return empty dataframe if no data
            
    # Perform the merges
    df_achat_recharge = pd.merge(
        achat_filtered, 
        rech_filtered,
        on=['msisdn', 'event_date'],
        how='outer',
        suffixes=('_achat', '_recharge')
    )
        
    # For the second merge, we need to handle the case where spin_filtered might be empty
    if not spin_filtered.empty:
        df_final = pd.merge(
            df_achat_recharge, 
            spin_filtered,
            left_on=['msisdn', 'event_date'],
            right_on=['msisdn', 'entry_date_hist'],
            how='outer'
        )
    else:
        df_final = df_achat_recharge.copy()
        # Add missing columns that would come from spin_filtered
        for col in spins_copy.columns:
            if col not in df_final.columns:
                df_final[col] = None
        
    #df_final=df_final[
    #    (df_final['msisdn'] == client_id) &
    #    (df_final['entry_date_hist'].between(date_debut, date_fin))
    #].copy()
    df_final['prix'] = abs(df_final['prix'])
    df_final['prix']=df_final['prix'].fillna(0)
    df_final['amount']=df_final['amount'].fillna(0)
    df_final["serv_orig"] = df_final["serv_orig"].str.replace(r"^_+", "", regex=True)
    df_final['spin_number'] = df_final['spin_number'].fillna(0)
    df_final['achat_option_duree'] = df_final['description'].apply(extract_duree)


    # S'assurer que la colonne est bien en entier
    df_final['achat_option_duree'] = df_final['achat_option_duree'].astype(int)
    df_final['type_client'] = df_final.apply(detect_type_client, axis=1)
    df_final = detect_maxit_usage(df_final)
    df_client = df_final.groupby("msisdn").agg({
    "type_client": lambda x: x.mode()[0] if not x.mode().empty else 'inconnu'
 
}).reset_index()
    df_client['nb_recharge'] = df_final['amount'][df_final['amount']>0].count()
    df_client['nb_achat'] = df_final['prix'][df_final['prix']>0].count()
    df_final['nombre_jeu'] = df_final['spin_number'].apply(get_nombre_jeu)
    nombre_jeu_par_client = df_final.groupby('msisdn')['nombre_jeu'].count()
    df_client = df_client.join(nombre_jeu_par_client.rename('nombre_jeu'), on='msisdn')
    df_client['nombre_jeu'] = df_client['nombre_jeu'].fillna(0).astype(int)
    df_final['interet'] = df_final['serv_orig'].apply(get_interet_type)
    interet_par_client = df_final.groupby('msisdn')['interet'].apply(get_mode)
    df_client = df_client.join(interet_par_client.rename('interet'), on='msisdn')
    # 1. Créer la colonne d'intérêt international (1 si "roaming" ou "inter" dans serv_orig, sinon 0)
    df_final['interet_international'] = df_final['serv_orig'].apply(
        lambda x: 1 if isinstance(x, str) and any(term in x.lower() for term in ['roaming', 'inter']) else 0
    )

    # 2. Prendre le maximum par client (au moins une occurrence => 1)
    interet_international_par_client = df_final.groupby('msisdn')['interet_international'].max()

    # 3. Joindre à df_global_seg en gardant l'alignement sur msisdn
    df_client = df_client.join(interet_international_par_client.rename('interet_international'), on='msisdn')
    df_client['interet_international'] = df_client['interet_international'].fillna(0).astype(int)

    
    # 1. Calculer la somme des achats par msisdn
    montant_total_achat = df_final.groupby('msisdn')['prix'].sum()

    # 2. Ajouter la colonne à df_global_seg en alignant sur msisdn
    df_client = df_client.join(montant_total_achat.rename('montant_total_achat'), on='msisdn')

    # 3. Remplacer les NaN par 0 pour les clients sans achat
    df_client['montant_total_achat'] = df_client['montant_total_achat'].fillna(0)
    
    result = df_final.groupby('msisdn').apply(check_client_maxit).reset_index()
    df_client = pd.merge(df_client, result, on='msisdn', how='left')
    # 1. Calculer la somme des achats par msisdn
    montant_total_recharge = df_final.groupby('msisdn')['amount'].sum()

    # 2. Ajouter la colonne à df_global_seg en alignant sur msisdn
    df_client = df_client.join(montant_total_recharge.rename('montant_total_recharge'), on='msisdn')

    # 3. Remplacer les NaN par 0 pour les clients sans achat
    df_client['montant_total_recharge'] = df_client['montant_total_recharge'].fillna(0)
    df_client['nb_achat'] = df_client['nb_achat'].fillna(0)
    df_client['nb_recharge'] = df_client['nb_recharge'].fillna(0)
    df_final['source_achat'] = df_final['serv_orig'].apply(get_source_achat)

    # 2. Take the maximum per client (at least one occurrence => 1)
    source_achat_par_client = df_final.groupby('msisdn')['source_achat'].max()
    df_client = df_client.join(source_achat_par_client.rename('source_achat'), on='msisdn')
    df_client['source_achat'] = df_client['source_achat'].fillna('sans achat')
    df_final['interet_promo'] = df_final['serv_orig'].apply(
    lambda x: 1 if isinstance(x, str) and 'promo' in x.lower() else 0
)

    interet_promo_par_client = df_final.groupby('msisdn')['interet_promo'].max()
    df_client = df_client.join(interet_promo_par_client.rename('interet_promo'), on='msisdn')
    df_client['interet_promo'] = df_client['interet_promo'].fillna(0).astype(int)
    df_client['type_usage'] = df_client.apply(get_type_usage, axis=1)
    df_client['action_majoritaire'] = df_client.apply(get_action, axis=1)
    df_client['montant_total_recharge'] = df_client['montant_total_recharge'].fillna(0)
    df_client['montant_total_achat'] = df_client['montant_total_achat'].fillna(0)
    # 1. Définir ton score d'engagement
    df_client['engagement_score'] = (
        df_client['nb_recharge'] +
        df_client['nb_achat'] +
        df_client['nombre_jeu']
    )

    # 2. Catégoriser en Low, Medium, High
    # Par exemple, baser sur les quantiles (qcut fait des coupures équilibrées)
    if df_client['engagement_score'].nunique() >= 3:
        df_client['engagement_level'] = pd.qcut(
            df_client['engagement_score'],
            q=3,
            labels=['Low', 'Medium', 'High'],
            duplicates='drop'
        )
    else:
        # If not enough unique values, assign all to 'Medium'
        df_client['engagement_level'] = 'Medium'
    # 1. Calculer la durée moyenne des options pour chaque client
    # Filtrer uniquement les lignes où achat_option_duree est > 0
    duree_moyenne_options=df_final [df_final['achat_option_duree'] > 0].groupby('msisdn')['achat_option_duree'].mean()

    # 2. Ajouter au df_global_seg
    df_client = df_client.join(duree_moyenne_options.rename('duree_moyenne_option'), on='msisdn')

    # 3. Remplacer les NaN par 0 pour les clients sans option
    df_client['duree_moyenne_option'] = df_client['duree_moyenne_option'].fillna(0)

    # 4. Convertir en entier pour éviter les valeurs décimales
    df_client['duree_moyenne_option'] = df_client['duree_moyenne_option'].astype(int)
    achat_majoritaire = df_final.groupby('msisdn')['description'].apply(get_major_achat)

    # 4. Ajouter au df_global_seg en utilisant join pour maintenir l'alignement
    df_client = df_client.join(achat_majoritaire.rename('achat_majoritaire'), on='msisdn')

    # 5. Remplacer les NaN par 'aucun achat'
    df_client['achat_majoritaire'] = df_client['achat_majoritaire'].fillna('aucun achat')
    total_achats = df_final.groupby('msisdn')['serv_orig'].count()

    # 2. Nombre d'achats en promo (en prenant en compte toutes les variations)
    achats_promo = df_final[df_final['serv_orig'].str.contains(
        r'promo|_promo|Promo|PROMO', case=False, na=False
    )].groupby('msisdn')['serv_orig'].count()

        # 3. Calculer le % de réactivité aux promotions et arrondir à 2 décimales
    taux_reponse_promo = (achats_promo / total_achats).fillna(0).round(2)

    # 4. Ajouter dans ton df global en utilisant join pour maintenir l'alignement
    df_client = df_client.join(taux_reponse_promo.rename('reponse_promo'), on='msisdn')

    # 5. Remplacer les NaN par 0 pour les clients sans achat
    df_client['reponse_promo'] = df_client['reponse_promo'].fillna(0)

    return df_client


In [ ]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np
import pandas as pd

for col in df_global_seg.columns:
        if df_global_seg[col].dtype == object:
            le = LabelEncoder()
            df_global_seg[col] = le.fit_transform(df_global_seg[col])
def normalize_and_pca(df, features):
    """
    Normalise les features et applique PCA

    ```
    Parameters:
    df (pd.DataFrame): Dataframe contenant les données
    features (list): Liste des features à normaliser

    Returns:
    pd.DataFrame: Dataframe normalisé
    """
    # Normalisation des features
    scaler = MinMaxScaler()
    df[features] = scaler.fit_transform(df[features])

    # Appliquer PCA
    pca = PCA()
    X_pca = pca.fit_transform(df[features])

    # Créer un nouveau dataframe avec les composantes principales
    df_pca = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])

    # Ajouter les colonnes non-numériques
    non_numeric_cols = [col for col in df.columns if col not in features]
    df_final = pd.concat([df[non_numeric_cols], df_pca], axis=1)

    return df_final, pca

import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

def find_best_k_using_elbow(data, k_range=range(1, 11), plot=True):
    inertias = []

    # Step 1: Calculate inertias for each k
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42)
        kmeans.fit(data)
        inertias.append(kmeans.inertia_)

    # Step 2: Find the "elbow" using the largest distance method
    k_vals = list(k_range)
    p1 = np.array([k_vals[0], inertias[0]])
    p2 = np.array([k_vals[-1], inertias[-1]])

    distances = []
    for i in range(len(k_vals)):
        p = np.array([k_vals[i], inertias[i]])
        # Use np.cross with 3D vectors to avoid deprecation warning
        cross_product = np.cross(np.array([p2[0] - p1[0], p2[1] - p1[1], 0]),
                               np.array([p1[0] - p[0], p1[1] - p[1], 0]))
        distance = np.linalg.norm(cross_product) / np.linalg.norm(np.array([p2[0] - p1[0], p2[1] - p1[1], 0]))
        distances.append(distance)

    best_k = k_vals[np.argmax(distances)]

    return best_k
segmentation_guide = {
        "rentabilité": ["rentabilite", "nb_achat", "nb_recharge", "frequence_utilisation_mensuelle", "moy_recharge", 'duree_moyenne_option'],
        "engagement": ['est_utilisateur_actif', 'frequence_utilisation_mensuelle', 'engagement_score', 'engagement_level'],
        "type_client": ['type_client', 'frequence_utilisation_mensuelle', 'type_usage'],
        "type_interet": ['type_usage', 'interet', 'derniere_offre_achetee'],
        "interet_international": ['type_usage', 'interet_international', 'source_achat'],
        "interet_jeu": ['type_client', 'nombre_essai_jeu', 'win_count'],
        "interet_promo": ['source_achat', 'interet_promo', 'type_usage', 'reponse_promo'],
        "action": ['nombre_jeu', 'nb_recharge', 'nb_achat', 'action_majoritaire']
    }

segments_labels = {
        'rentabilité': {'rentabilite': ['non rentable', 'rentable']},
        'engagement': {'engagement_score': ['non engagé', 'peu engagé', 'très engagé']},
        "type_client": {'type_client': ['orienté USSD', 'orienté APLICATION', 'orienté BOUTIQUE']},
        "type_interet": {'interet': ['data', 'voix']},
        "interet_international": {'interet_international': ['non international', 'international']},
        "interet_jeu": {'nombre_essai_jeu': ['non jeu', 'peu jeu', 'très jeu']},
        "interet_promo": {'interet_promo': ['non promo', 'peu promo', 'Sensibles aux promos']},
        "action": {'action_majoritaire': ['achat', 'recharge', 'roue chance']}
    }
def assign_labels_by_feature(centroids, feature_index, ordered_labels):
    """
    Attribue des labels aux clusters selon la valeur d'une seule feature choisie.

    Args:
    centroids (np.ndarray): Matrice (n_clusters x n_features)
    feature_index (int): Index de la feature à utiliser pour le tri (ex: 0 pour 1re colonne)
    ordered_labels (list of str): Labels à assigner dans l’ordre croissant de la feature

    Returns:
    dict: Mapping des index de cluster vers labels
    """

    # Extraire la colonne (feature) choisie
    feature_values = centroids[:, feature_index]

    # Trier les indices de clusters selon cette feature
    sorted_indices = np.argsort(feature_values)

    # Assigner les labels dans l’ordre souhaité
    label_mapping = {idx: ordered_labels[i] for i, idx in enumerate(sorted_indices)}

    return label_mapping

def preprocess_dataframe(df, threshold=0.1):
    # Create a copy of the DataFrame to avoid SettingWithCopyWarning
    df = df.copy()
    
    # Label encode categorical columns
    for col in df.columns:
        if df[col].dtype == object:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
    
    # Select numerical columns (excluding 'msisdn')
    numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    numerical_cols = [col for col in numerical_cols if col != 'msisdn']

    # Initialize Min-Max scaler
    scaler = MinMaxScaler()

    # List to store features that need normalization
    features_to_normalize = []

    # Check differences between min and max values for each feature
    for col in numerical_cols:
        min_val = df[col].min()
        max_val = df[col].max()

        # Calculate relative difference
        diff = (max_val - min_val) / min_val if min_val != 0 else max_val

        # If difference is greater than threshold, add feature to list
        if diff > threshold:
            features_to_normalize.append(col)

    # Apply Min-Max normalization if necessary
    if features_to_normalize:
        print(f"Normalisation des features suivantes: {features_to_normalize}")
        df[features_to_normalize] = scaler.fit_transform(df[features_to_normalize])
    else:
        print("Aucune normalisation nécessaire, les échelles des features sont déjà similaires.")

    return df

from sklearn.cluster import KMeans
def segmenter_clients(df, critere_segmentation, acquisition=True):
    # Filtrage initial
    
    
    # Dictionnaires
    segmentation_guide={
    "rentabilité":["rentabilite","nb_achat","nb_recharge","frequence_utilisation_mensuelle","moy_recharge",'duree_moyenne_option'],
    "engagement":['est_utilisateur_actif','frequence_utilisation_mensuelle','engagement_score','engagement_level'],
    "type_client":['type_client','frequence_utilisation_mensuelle','type_usage'],
    "type_interet":['interet','duree_moyenne_option'],
    "interet_international":['type_usage','interet_international','source_achat','interet'],
    "interet_jeu":['nombre_jeu','win_count'],
    "interet_promo":['interet_promo','reponse_promo',],
    "action":['nombre_jeu','nb_recharge', 'nb_achat','action_majoritaire']
}
    segments_labels = {
    'rentabilité': {
        'rentabilite': ['non rentable', 'rentable']
    },
    'engagement': {
        'engagement_score': ['non engagé','peu engagé', 'très engagé']
    },
    "type_client":{
        'type_client': ['orienté USSD', 'orienté APLICATION', 'orienté BOUTIQUE']
    },
    "type_interet":{  
        'interet':['data','voix']},
    "interet_international":{
        'interet_international':['non international', 'international']
    },
    "interet_jeu":{
        'nombre_jeu':['non jeu','peu jeu','très jeu']
    },
    "interet_promo":{
        'interet_promo':['non promo','peu promo','Sensibles aux promos']
    },
    "action":{
        'action_majoritaire':['achat','recharge','roue chance']
    }
}

    # Sélection des features
    df=df.copy()
    if acquisition:
        df=df[df['type_client'] != 'maxit']
    else:
        df=df[df['type_client'] == 'maxit']
    features = segmentation_guide[critere_segmentation] + ['msisdn']
    df_seg = df[features].copy()
    df_seg = preprocess_dataframe(df_seg.copy())

    # Supprimer msisdn avant clustering
    if 'msisdn' in df_seg.columns:
        df_seg.drop(columns=['msisdn'], inplace=True)

    # Trouver l'attribut principal de segmentation
    feature_name = list(segments_labels[critere_segmentation].keys())[0]
    ordered_labels = segments_labels[critere_segmentation][feature_name]
    feature_index = df_seg.columns.get_loc(feature_name)

    # Réduction dimensionnelle optionnelle
    # df_seg, _ = select_features_pca(df_seg)  # Décommenter si défini

    # Détermination du meilleur k
    best_k = find_best_k_using_elbow(df_seg)

    # Clustering
    model = KMeans(init='k-means++', n_clusters=best_k, max_iter=500, random_state=22)
    segments = model.fit_predict(df_seg)

    # Affectation des résultats
    df['segments'] = segments
    centroids = model.cluster_centers_
    mapping = assign_labels_by_feature(centroids, feature_index, ordered_labels)
    df[f'segment_{critere_segmentation}'] = df['segments'].map(mapping)

    return df

for critere_segmentation in segmentation_guide.keys():
    print("segmentation selon ", critere_segmentation)
    df_global_seg = segmenter_clients(df_global_seg, critere_segmentation, acquisition=True)